# Ensemble Methods — The Math Behind Majority Voting

## What is an Ensemble?

An ensemble combines the predictions of multiple models to get a better result than any single model alone.

The intuition is the **wisdom of the crowd**:
> If you ask 1000 people to guess the weight of a cow,
> the average of their guesses is usually more accurate than any single expert.

In ML, instead of one model making a decision, many models vote.
The majority wins.

## Two types of ensembles

| Type | Models used | Example |
|------|------------|--------|
| **Homogeneous** | Same model type, different training data | Bagging, Random Forest |
| **Heterogeneous** | Different model types, same training data | Voting (LR + DT + KNN) |

## What this notebook covers

We use **probability theory** to understand exactly why and when ensembles work:

| Section | Question answered |
|---------|------------------|
| 1 | If each model has accuracy p, what is the ensemble accuracy? |
| 2 | When does adding more models help vs hurt? |
| 3 | How does accuracy grow as we add more models? |
| 4 | How do heterogeneous models combine? |
| 5 | Real demo with actual classifiers |

## Step 1 — The Binomial Model of Majority Voting

Suppose we have **n independent models**, each with the **same accuracy p**.
Each model independently answers correctly with probability p, or wrongly with probability (1-p).

The ensemble uses **majority vote** — it is correct when more than half the models are correct.

The probability that exactly **k out of n** models are correct follows the **binomial distribution**:

$$
P(\text{exactly } k \text{ correct}) = \binom{n}{k} \cdot p^k \cdot (1-p)^{n-k}
$$

The ensemble is correct when k > n/2, so:

$$
P(\text{ensemble correct}) = \sum_{k > n/2} \binom{n}{k} \cdot p^k \cdot (1-p)^{n-k}
$$

We compute this sum for different values of p and n.

In [ ]:
import numpy as np
import pandas as pd
import math
import itertools
import matplotlib.pyplot as plt

def ensemble_accuracy(p, n):
    """
    Compute theoretical ensemble accuracy using majority vote.

    p : accuracy of each individual model (float between 0 and 1)
    n : number of models (odd integer recommended to avoid ties)

    Returns a DataFrame showing the breakdown by k (number of correct models),
    and prints the total ensemble accuracy.
    """
    rows = []
    for k in range(n + 1):
        prob        = math.comb(n, k) * (p**k) * ((1-p)**(n-k))
        is_majority = k > n / 2
        rows.append({
            'k (models correct)':     k,
            'P(exactly k correct)':   round(prob, 5),
            'Majority vote correct?': is_majority,
            'Contribution':           round(prob, 5) if is_majority else 0
        })

    df = pd.DataFrame(rows)
    total = df['Contribution'].sum()
    print(f'Individual model accuracy:  {p}')
    print(f'Number of models:           {n}')
    print(f'Ensemble accuracy:          {round(total, 5)}')
    return df

### Example A — Individual accuracy = 0.7 (better than random)

Each model gets 70% of examples right. What does a 5-model ensemble get?

In [ ]:
ensemble_accuracy(p=0.7, n=5)

### Example B — Individual accuracy = 0.51 (barely better than random)

Each model is only slightly better than guessing.
With enough models, the ensemble can still be quite reliable.

In [ ]:
ensemble_accuracy(p=0.51, n=101)

### Example C — Individual accuracy = 0.4 (worse than random)

This is the counterintuitive case. If each model is wrong more than right,
the ensemble is **worse** than any individual model.
Majority vote amplifies the wrong answer.

In [ ]:
ensemble_accuracy(p=0.4, n=5)

## Step 2 — The Critical Rule: p Must Be > 0.5

We plot ensemble accuracy for different individual accuracies.
This shows the key rule clearly:

> **If p > 0.5**, adding more models always helps — ensemble accuracy is higher than individual accuracy.
> **If p < 0.5**, adding more models always hurts — ensemble accuracy is lower than individual accuracy.
> **If p = 0.5**, the ensemble stays at 0.5 no matter how many models you add.

This is why model quality matters — ensembling bad models makes them worse.

In [ ]:
def ensemble_acc_value(p, n):
    return sum(
        math.comb(n, k) * (p**k) * ((1-p)**(n-k))
        for k in range(n+1) if k > n/2
    )

p_values = np.linspace(0.3, 0.9, 200)

plt.figure(figsize=(8, 5))
for n in [3, 5, 11, 51]:
    ens_accs = [ensemble_acc_value(p, n) for p in p_values]
    plt.plot(p_values, ens_accs, label=f'n = {n} models')

plt.plot(p_values, p_values, 'k--', label='Individual accuracy (baseline)', linewidth=1.5)
plt.axvline(0.5, color='gray', linestyle=':', linewidth=1)
plt.axhline(0.5, color='gray', linestyle=':', linewidth=1)
plt.fill_betweenx([0, 1], 0.3, 0.5, alpha=0.05, color='red',   label='p < 0.5: ensemble hurts')
plt.fill_betweenx([0, 1], 0.5, 0.9, alpha=0.05, color='green', label='p > 0.5: ensemble helps')
plt.title('Ensemble Accuracy vs Individual Accuracy')
plt.xlabel('Individual model accuracy (p)')
plt.ylabel('Ensemble accuracy')
plt.legend(fontsize=9)
plt.grid(True)
plt.tight_layout()
plt.show()

## Step 3 — How Many Models Do You Need?

For a fixed individual accuracy p, we plot ensemble accuracy as we add more models.

The ensemble improves quickly at first, then the gains diminish.
There is a point of diminishing returns — doubling the models gives half the benefit.

In [ ]:
n_values = range(1, 102, 2)  # odd numbers only to avoid ties

plt.figure(figsize=(8, 5))
for p in [0.6, 0.7, 0.8, 0.9]:
    ens_accs = [ensemble_acc_value(p, n) for n in n_values]
    plt.plot(list(n_values), ens_accs, label=f'p = {p}')

plt.title('Ensemble Accuracy vs Number of Models')
plt.xlabel('Number of models (n)')
plt.ylabel('Ensemble accuracy')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

print('Diminishing returns — going from 1 to 11 models helps a lot.')
print('Going from 51 to 101 models helps very little.')

## Step 4 — Heterogeneous Models: All Outcome Combinations

When the models have **different accuracies** (e.g. LR=0.8, DT=0.7, KNN=0.65),
we can no longer use the binomial formula (which assumes all models are identical).

Instead, we enumerate **every possible combination** of correct/wrong outcomes
and compute their joint probability.

For 3 models there are 2³ = 8 combinations.
For each combination we check if the majority got it right.

The ensemble accuracy = sum of probabilities of all combinations where majority is correct.

In [ ]:
def heterogeneous_ensemble(probabilities):
    """
    Enumerate all correct/wrong combinations for heterogeneous models.

    probabilities : list of individual model accuracies
    Returns a DataFrame of all combinations with joint probability,
    and prints the ensemble accuracy.
    """
    n = len(probabilities)
    outcomes = list(itertools.product(['Correct', 'Wrong'], repeat=n))

    rows = []
    for combo in outcomes:
        joint_prob = 1.0
        for outcome, p in zip(combo, probabilities):
            joint_prob *= p if outcome == 'Correct' else (1 - p)

        n_correct    = combo.count('Correct')
        majority_win = n_correct > n / 2

        row = {f'Model {i+1} (p={probabilities[i]})': combo[i] for i in range(n)}
        row['Joint Probability'] = round(joint_prob, 5)
        row['Majority correct?'] = majority_win
        rows.append(row)

    df = pd.DataFrame(rows)
    ensemble_acc = df.loc[df['Majority correct?'], 'Joint Probability'].sum()

    print(f'Individual accuracies: {probabilities}')
    print(f'Ensemble accuracy:     {round(ensemble_acc, 5)}')
    print(f'Best individual:       {max(probabilities)}')
    return df

heterogeneous_ensemble([0.8, 0.7, 0.65])

### When does heterogeneous ensemble beat the best individual model?

The ensemble wins when models make **different mistakes**.
If all models fail on the same examples, voting does not help.

Below we compare three scenarios:
- Models with similar accuracy (all strong)
- Models with varying accuracy (mixed)
- One strong model + two weak models

In [ ]:
scenarios = [
    ([0.8, 0.8, 0.8], 'All equal (0.8 each)'),
    ([0.9, 0.7, 0.6], 'Mixed (0.9, 0.7, 0.6)'),
    ([0.9, 0.55, 0.55], 'One strong + two weak'),
    ([0.6, 0.6, 0.6, 0.6, 0.6], '5 models at 0.6 each'),
]

print(f'{"Scenario":<35} {"Best Individual":>16} {"Ensemble":>10} {"Gain":>8}')
print('-' * 73)
for probs, label in scenarios:
    n = len(probs)
    outcomes = list(itertools.product(['C', 'W'], repeat=n))
    ens_acc = 0.0
    for combo in outcomes:
        joint = 1.0
        for o, p in zip(combo, probs):
            joint *= p if o == 'C' else (1-p)
        if combo.count('C') > n/2:
            ens_acc += joint
    best = max(probs)
    print(f'{label:<35} {best:>16.3f} {ens_acc:>10.3f} {ens_acc-best:>+8.3f}')

## Step 5 — Real Demo: Hard Voting on Breast Cancer Dataset

We now use actual classifiers to verify the theory on real data.
Hard voting = each model gets one vote, majority wins.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

data  = load_breast_cancer()
X, y  = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Three different model types
models = [
    ('Logistic Regression', LogisticRegression(max_iter=5000)),
    ('Decision Tree',       DecisionTreeClassifier(max_depth=4, random_state=42)),
    ('KNN',                 KNeighborsClassifier(n_neighbors=7)),
]

# Train each model and collect test predictions
all_preds = []
print(f'{'Model':<25}  Accuracy')
print('-' * 36)
for name, model in models:
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    all_preds.append(preds)
    print(f'{name:<25}  {accuracy_score(y_test, preds):.4f}')

# Hard vote — majority of 3 models
votes     = np.array(all_preds)          # shape (3, n_test)
hard_vote = (votes.mean(axis=0) >= 0.5).astype(int)

print('-' * 36)
print(f'{'Hard Vote Ensemble':<25}  {accuracy_score(y_test, hard_vote):.4f}')

In [ ]:
# Show where models agree and disagree
votes_df = pd.DataFrame(
    np.array(all_preds).T,
    columns=['LR', 'DT', 'KNN']
)
votes_df['Ensemble'] = hard_vote
votes_df['True']     = y_test
votes_df['All agree?'] = (votes_df[['LR','DT','KNN']].nunique(axis=1) == 1)

n_agree    = votes_df['All agree?'].sum()
n_disagree = (~votes_df['All agree?']).sum()
print(f'Samples where all 3 models agree:    {n_agree}  ({100*n_agree/len(votes_df):.0f}%)')
print(f'Samples where models disagree:       {n_disagree}  ({100*n_disagree/len(votes_df):.0f}%)')
print()
print('First 10 predictions:')
print(votes_df.head(10).to_string(index=False))

## Summary

### The Math
| Concept | Formula |
|---------|--------|
| P(exactly k of n models correct) | C(n,k) * p^k * (1-p)^(n-k) |
| Ensemble accuracy (homogeneous) | sum of above for all k > n/2 |
| Ensemble accuracy (heterogeneous) | sum of joint probabilities for majority-correct combos |

### The Rules

| Condition | Effect |
|-----------|--------|
| Individual accuracy p > 0.5 | Ensemble accuracy > individual accuracy |
| Individual accuracy p < 0.5 | Ensemble accuracy < individual accuracy |
| More models (if p > 0.5) | Accuracy increases, but with diminishing returns |
| Models make different mistakes | Ensemble gains are larger |
| Models all fail on same examples | Ensemble gains are minimal |

### Key Insight

> **Ensemble methods work because errors cancel out.**
> Each model makes different mistakes on different examples.
> The majority vote is right even when individual models are wrong —
> as long as each model is right more often than wrong (p > 0.5).

### Connection to Other Notebooks
- **Bagging** — homogeneous ensemble (same model, different bootstrap samples)
- **Random Forest** — bagging + feature randomness
- This notebook shows the **theoretical foundation** that makes all of them work